# Border and Area Computation
This notebook allows you to:
- Calculate the borders and areas of organoids for each image in the input folder.
- Visualize individual images with their corresponding borders and areas.

In [ ]:
# Import necessary libraries
import os
import pandas as pd
import matplotlib.image as mpimg
from cellpose import io, models
from skimage import measure, segmentation
import ipywidgets as widgets
from IPython.display import display, clear_output
from tqdm.notebook import tqdm
import plotly.graph_objs as go
from PIL import Image


In [ ]:
# Function to calculate areas of segmented regions
def calculate_areas(mask):
    labeled_mask = measure.label(mask)
    region_props = measure.regionprops(labeled_mask)
    areas = [(region.label, region.area, region.centroid) for region in region_props]
    return areas

In [ ]:
# Segmentation function
def segment_images(image_folder, output_folder):
    model = models.Cellpose(gpu=False, model_type="cyto")

    os.makedirs(output_folder, exist_ok=True)

    image_files = [f for f in os.listdir(image_folder) if f.lower().endswith((".png", ".jpg", ".jpeg", ".tif", ".tiff"))]

    # Initialize progress bar
    progress = tqdm(total=len(image_files), desc="Processing images", leave=False)

    for image_name in image_files:
        image_path = os.path.join(image_folder, image_name)
        image = io.imread(image_path)

        masks, _, _, _ = model.eval(image, diameter=None, channels=[0, 0])
        areas = calculate_areas(masks)

        # Create an image with segmented borders and IDs
        bordered_image = segmentation.mark_boundaries(image, masks, outline_color=(1, 0, 0))

        output_path_borders = os.path.join(output_folder, f"{image_name}")
        mpimg.imsave(output_path_borders, bordered_image)

        # Save areas to a separate CSV file
        area_data = [{"id": label, "area": area, "x": centroid[0], "y": centroid[1]} for label, area, centroid in areas]
        area_df = pd.DataFrame(area_data)
        area_csv_path = os.path.join(output_folder, f"{os.path.splitext(image_name)[0]}_areas.csv")
        area_df.to_csv(area_csv_path, index=False)

        # Update progress bar
        progress.update(1)

    progress.close()

## Step 1: Calculate borders of organoids
1. Specify the input folder containing images of organoids (default: organoid_images).
2. Specify the output folder where the results will be saved.
3. Click the 'Start Calculation' button to process the images.

In [ ]:
# Define the folder input widgets
input_folder_input = widgets.Text(
    value='organoid_images',
    placeholder='Insert path here',
    description='Source images folder:',
    disabled=True,
    style= {'description_width': 'initial'},
    layout=widgets.Layout(width='50%')
)
checkbox_input_folder = widgets.Checkbox(
    value=False,
    description='edit path',
    disabled=False,
    indent=False
)

def input_folder_change(c):
    if checkbox_input_folder.value:
        input_folder_input.disabled = False
    else:
        input_folder_input.disabled = True

checkbox_input_folder.observe(input_folder_change, names="value")

output_folder_input = widgets.Text(
    value='bordered_organoid_images',
    placeholder='Insert path here',
    description='Output images folder:',
    disabled=True,
    style= {'description_width': 'initial'},
    layout=widgets.Layout(width='50%')
)
checkbox_output_folder = widgets.Checkbox(
    value=False,
    description='edit path',
    disabled=False,
    indent=False
)

def output_folder_change(c):
    if checkbox_output_folder.value:
        output_folder_input.disabled = False
    else:
        output_folder_input.disabled = True

checkbox_output_folder.observe(output_folder_change, names="value")


# Define the button to start segmentation
start_button = widgets.Button(
    description='Start Calculation',
    disabled=False,
    button_style='success',
    tooltip='Click to start the calculation',
    icon='check'
)

# Define the output widgets
segmentation_output = widgets.Output()

# Define the function to be called when the button is clicked
def on_button_click(b):
    with segmentation_output:
        clear_output()
        segment_images(input_folder_input.value, output_folder_input.value)
        update_image_list()
        display(widgets.HTML(value=f"<b>Processing Completed!</b>"))
        display(widgets.HTML(value=f"The results are stored in the <b>{output_folder_input.value}</b> folder."))
        display(widgets.HTML(value=f"For each image in the <b>{input_folder_input.value}</b> folder, a new image with borders and a corresponding CSV file detailing the areas are saved."))

# Bind the button click event to the function
start_button.on_click(on_button_click)

input_folder_section = widgets.HBox([input_folder_input, checkbox_input_folder])
output_folder_section = widgets.HBox([output_folder_input, checkbox_output_folder])
input_section = widgets.VBox([input_folder_section, output_folder_section, start_button])
display(widgets.VBox([input_section, segmentation_output]))


## Step 2: Visualize bordered images with areas
Select an image from the dropdown to view the images with borders. Hover over each organoid to view its area.

In [ ]:
def get_bordered_images(image_folder):
    if os.path.exists(image_folder):
        return [f for f in os.listdir(image_folder) if f.lower().endswith((".png", ".jpg", ".jpeg", ".tif", ".tiff"))]
    return []

def update_image_list(change=None):
    image_files = get_bordered_images(output_folder_input.value)
    image_select.options = image_files

output_folder_input.observe(update_image_list, names='value')

# Define the image selection widgets
image_select = widgets.Dropdown(
    options= [],
    description='Select Image:',
    style= {'description_width': 'initial'},
    layout=widgets.Layout(width='50%'),
)

# Define the output widgets
visualization_output = widgets.Output()

def on_image_select_change(change):
    with visualization_output:
        clear_output()
        global fig
        # Load your image
        image_name = image_select.value
        image_path = os.path.join(output_folder_input.value, image_name)
        image = Image.open(image_path)
        
        # Apply a scaling factor to make the image smaller
        scaling_factor = 0.5
        scaled_width = int(image.size[0] * scaling_factor)
        scaled_height = int(image.size[1] * scaling_factor)
        
        # Resize the image
        image = image.resize((scaled_width, scaled_height))

        df = pd.read_csv(f"{os.path.splitext(image_path)[0]}_areas.csv")

        # Create a plotly figure
        fig = go.Figure()

        # Add the image to the figure
        fig.add_layout_image(
            dict(
                source=image,
                xref="x",
                yref="y",
                x=0,
                y=scaled_height,  # adjust for the height of the image
                sizex=scaled_width,
                sizey=scaled_height,
                sizing="stretch",
                opacity=1,
                layer="below"
            )
        )

        # Add scatter plot for centroids
        fig.add_trace(
            go.Scatter(
                x=df['y'] * scaling_factor,
                y=[scaled_height - (x * scaling_factor) for x in df['x']], 
                mode='text',  # show only the text
                text=df['id'],  # use df['id'] for the labels
                textposition='top center',  # position the text above the markers
                hovertemplate='ID: %{text}<br>Area: %{customdata}<extra></extra>',
                customdata=df['area']  # include area information for hover
            )
        )

        # Update layout to match image dimensions
        fig.update_layout(
            xaxis=dict(
                visible=False,
                range=[0, scaled_width]
            ),
            yaxis=dict(
                visible=False,
                range=[0, scaled_height]
            ),
            width=scaled_width,
            height=scaled_height,
            margin=dict(l=0, r=0, t=0, b=0)
        )

        fig.show()

image_select.observe(on_image_select_change, names='value')

display(image_select)
display(visualization_output)